In [12]:
import pandas as pd
import torch
import os
import tqdm
from conllu import parse
from torch.utils.data.dataset import Dataset
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from torch.utils.data import DataLoader
from torch.optim import AdamW
from sklearn.metrics import hamming_loss, f1_score, classification_report

In [13]:
def load_and_map_labels(label_file_path):
    # Load the label file
    labels_df = pd.read_csv(label_file_path, sep="\t", header=None, names=["article_id", "narratives", "subnarratives"])
    
    # Map article_id to labels
    labels_mapping = {
        row["article_id"]: {
            "narratives": row["narratives"].split(";"),
            "subnarratives": row["subnarratives"].split(";")
        }
        for _, row in labels_df.iterrows()
    }
    
    return labels_mapping

label_file = "../data/training_data_16_October_release/EN/subtask-2-annotations.txt"
labels_mapping = load_and_map_labels(label_file)
print(labels_mapping["EN_UA_015443.txt"])

{'narratives': ['URW: Discrediting the West, Diplomacy', 'URW: Amplifying war-related fears', 'URW: Amplifying war-related fears'], 'subnarratives': ['URW: Discrediting the West, Diplomacy: West is tired of Ukraine', 'URW: Amplifying war-related fears: NATO should/will directly intervene', 'URW: Amplifying war-related fears: By continuing the war we risk WWIII']}


In [14]:
def parse_conllu_file(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        data = f.read()
    token_lists = parse(data)
    all_tokens = [token["form"] for token_list in token_lists for token in token_list]
    return " ".join(all_tokens)

In [15]:
def load_articles_from_conllu(articles_path, article_ids, labels_mapping):
    articles_data = []
    for article_id in article_ids:
        file_path = os.path.join(articles_path, f"{article_id.replace('.txt', '.conllu')}")
        if os.path.exists(file_path):
            article_text = parse_conllu_file(file_path)
            labels = labels_mapping.get(article_id, {"narratives": [], "subnarratives": []})
            articles_data.append({
                "article_id": article_id,
                "text": article_text,
                "narratives": labels["narratives"],
                "subnarratives": labels["subnarratives"]
            })
    return articles_data

# Load the dataset
articles_path = "../data/tmp/EN"
article_ids = labels_mapping.keys()
articles_data = load_articles_from_conllu(articles_path, article_ids, labels_mapping)

In [16]:
# Initialize tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

# Tokenize the data
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

tokenized_data = [tokenize_function(article) for article in articles_data]

In [17]:
# Generate label vocab
all_narratives = sorted({n for article in articles_data for n in article["narratives"]})
all_subnarratives = sorted({sn for article in articles_data for sn in article["subnarratives"]})

narrative_to_index = {n: i for i, n in enumerate(all_narratives)}
subnarrative_to_index = {sn: i for i, sn in enumerate(all_subnarratives)}

# Encode labels
def encode_labels(narratives, subnarratives):
    narrative_vector = [0] * len(all_narratives)
    subnarrative_vector = [0] * len(all_subnarratives)

    for n in narratives:
        narrative_vector[narrative_to_index[n]] = 1
    for sn in subnarratives:
        subnarrative_vector[subnarrative_to_index[sn]] = 1

    return narrative_vector + subnarrative_vector

for article in articles_data:
    article["labels"] = encode_labels(article["narratives"], article["subnarratives"])


In [18]:
class NarrativeDataset(Dataset):
    def __init__(self, articles):
        self.articles = articles

    def __len__(self):
        return len(self.articles)

    def __getitem__(self, idx):
        article = self.articles[idx]
        inputs = tokenizer(
            article["text"],
            padding="max_length",
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )
        labels = torch.tensor(article["labels"], dtype=torch.float)
        return {**inputs, "labels": labels}


In [19]:
def get_predictions(model, data_loader, device, threshold=0.05):
    model.eval()
    all_predictions = []
    all_labels = []
    with torch.no_grad():
        for batch in data_loader:
            # Move data to device
            inputs = {key: val.squeeze(1).to(device) for key, val in batch.items() if key != "labels"}
            labels = batch["labels"].to(device)
            
            # Forward pass
            outputs = model(**inputs)
            logits = outputs.logits
            probs = torch.sigmoid(logits)  # Convert logits to probabilities
            preds = (probs > threshold).int()  # Apply threshold for binary classification
            
            all_predictions.append(preds.cpu())
            all_labels.append(labels.cpu())
    
    # Concatenate predictions and labels
    all_predictions = torch.cat(all_predictions, dim=0)
    all_labels = torch.cat(all_labels, dim=0)
    return all_predictions.numpy(), all_labels.numpy()

In [20]:
def evaluate_model(model, data_loader, device, val_data, class_labels, print_report=True):
    y_pred, y_true = get_predictions(model, data_loader, device)

    # Hamming Loss
    hamming = hamming_loss(y_true, y_pred)

    # Macro and Micro F1 Scores
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)
    micro_f1 = f1_score(y_true, y_pred, average='micro', zero_division=0)

    # Subset Accuracy
    subset_accuracy = (y_true == y_pred).all(axis=1).mean()

    if print_report:
        report = classification_report(
            y_true, y_pred, target_names=class_labels, digits=2, zero_division=0
        )
        print("\nClassification Report:\n")
        print(report)

    return {"Hamming Loss": hamming, "Macro F1": macro_f1, "Micro F1": micro_f1, "Subset Accuracy": subset_accuracy}


In [21]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model
num_labels = len(all_narratives) + len(all_subnarratives)
model = BertForSequenceClassification.from_pretrained(
    "bert-base-multilingual-cased", 
    num_labels=num_labels
)
model = model.to(device)
model.train()

# Train-Validation Split
train_data, val_data = train_test_split(articles_data, test_size=0.2, random_state=42)
train_dataset = NarrativeDataset(train_data)
val_dataset = NarrativeDataset(val_data)

train_loader = DataLoader(train_dataset, batch_size=6, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=6, shuffle=False, pin_memory=True)

# Optimizer
optimizer = AdamW(model.parameters())

# Training Loop
num_epochs = 3
for epoch in range(num_epochs):  # Adjust epoch count as needed
    print(f"Epoch {epoch + 1}/{num_epochs}")
    epoch_loss = 0
    progress_bar = tqdm.tqdm(train_loader, desc="Processing Batches", leave=True)
    
    for batch in progress_bar:
        optimizer.zero_grad()
        
        # Ensure all tensors are moved to the correct device
        inputs = {key: val.squeeze(1).to(device) for key, val in batch.items() if key != "labels"}
        labels = batch["labels"].to(device)
        
        # Forward pass
        outputs = model(**inputs, labels=labels)
        loss = outputs.loss
        
        # Backward pass and optimization
        loss.backward()
        optimizer.step()
        
        # Update loss and progress bar
        epoch_loss += loss.item()
        progress_bar.set_postfix({"Batch Loss": f"{loss.item():.4f}"})
    
    # Print average loss for the epoch
    print(f"Epoch {epoch + 1} completed.")
    print(f"- Average Loss: {epoch_loss / len(train_loader):.4f}")
    
    # check that validation loss is not increasing
    val_metrics = evaluate_model(model, val_loader, device, val_data, all_narratives + all_subnarratives, print_report=False)
    print(f"- Validation: F1: {val_metrics['Micro F1']:.4f} | Hamming Loss: {val_metrics['Hamming Loss']:.4f}")


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1/3


Processing Batches: 100%|██████████| 27/27 [00:12<00:00,  2.20it/s, Batch Loss=0.0942]


Epoch 1 completed.
- Average Loss: 0.2251
- Validation: F1: 0.1672 | Hamming Loss: 0.2734
Epoch 2/3


Processing Batches: 100%|██████████| 27/27 [00:12<00:00,  2.25it/s, Batch Loss=0.1392]


Epoch 2 completed.
- Average Loss: 0.1395
- Validation: F1: 0.2057 | Hamming Loss: 0.1658
Epoch 3/3


Processing Batches: 100%|██████████| 27/27 [00:12<00:00,  2.22it/s, Batch Loss=0.1889]


Epoch 3 completed.
- Average Loss: 0.1428
- Validation: F1: 0.1856 | Hamming Loss: 0.1788


In [22]:
print("Evaluating on Validation Set...")
evaluation_results = evaluate_model(model, val_loader, device, val_data, all_narratives + all_subnarratives)

Evaluating on Validation Set...

Classification Report:

                                                                                                            precision    recall  f1-score   support

                                                                          CC: Climate change is beneficial       0.00      0.00      0.00         1
                                                                  CC: Controversy about green technologies       0.00      0.00      0.00         3
                                                                         CC: Criticism of climate movement       0.10      1.00      0.18         4
                                                                         CC: Criticism of climate policies       0.00      0.00      0.00         4
                                                             CC: Criticism of institutions and authorities       0.12      1.00      0.22         5
                                                      